# Week 4 — PhoBERT + LLM Augmentation & Explainability
**ABSA VLSP 2018 Hotel | NLP Course — HUST**

Pipeline:
```
Phase 1 (đã chạy local): LLM sinh data → filter → train_augmented.csv (commit lên Git)
Phase 1c (Cell 6)       : PhoBERT re-train trên augmented data  ← CẦN GPU
Phase 2 (Cell 7–10)     : PhoBERT predict → LLM giải thích kết quả
```

> Chạy theo thứ tự Cell 1 → Cell 10.

In [ ]:
# ============================================================
# Cell 1 — Check GPU & Install dependencies
# ============================================================
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}')
    print(f'VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('Khong co GPU! Kaggle: Settings -> Accelerator -> GPU T4 x2')

torch.cuda.empty_cache()
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

# Dependencies (transformers + underthesea đã có từ week 2 run)
!pip install -q transformers==4.38.0 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece
!pip install -q openai google-generativeai
print('Dependencies installed')

In [ ]:
# ============================================================
# Cell 2 — Clone repo từ GitHub & Setup working directory
# ============================================================
import os, sys

REPO_URL    = 'https://github.com/vudinhminh08/NLP-project-master-study.git'
REPO_BRANCH = 'master'
PROJECT_DIR = '/kaggle/working/absa-project'

if not os.path.exists(PROJECT_DIR):
    print(f'Cloning {REPO_URL} ...')
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
    print('Clone xong')
else:
    print(f'Repo da ton tai — pulling latest...')
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

for d in ['outputs/models', 'outputs/results', 'outputs/results/week4_augmented',
          'outputs/results/week4_augmented/models', 'outputs/eda']:
    os.makedirs(d, exist_ok=True)

for p in ['code/week1', 'code/week2', 'code/week3', 'code/week3_part2']:
    sys.path.insert(0, p)

print('Paths added:', ['code/week1', 'code/week2', 'code/week3', 'code/week3_part2'])

# Verify data augmented da co
for f in ['data/train_preprocessed.csv', 'data/dev_preprocessed.csv',
          'data/test_preprocessed.csv', 'data/train_augmented.csv']:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  [{status}] {f}')

In [ ]:
# ============================================================
# Cell 3 — API Keys (dung Kaggle Secrets)
# ============================================================
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

PROVIDER = 'openai'   # hoac 'gemini'

try:
    OPENAI_API_KEY = secrets.get_secret('OPENAI_API_KEY')
    print('OPENAI_API_KEY loaded from Kaggle Secrets')
except Exception:
    OPENAI_API_KEY = ''  # fallback: dien truc tiep
    print('OPENAI_API_KEY not in Secrets')

try:
    GEMINI_API_KEY = secrets.get_secret('GEMINI_API_KEY')
    print('GEMINI_API_KEY loaded from Kaggle Secrets')
except Exception:
    GEMINI_API_KEY = ''

API_KEY = OPENAI_API_KEY if PROVIDER == 'openai' else GEMINI_API_KEY

if not API_KEY:
    print('WARNING: API key trong: Add-ons -> Secrets')
else:
    print(f'Provider: {PROVIDER} | API key: OK')

In [ ]:
# ============================================================
# Cell 4 — Verify config & data
# ============================================================
import json
import pandas as pd
from utils.constants import RARE_ASPECTS, ASPECT_COLUMNS
from utils.helpers import set_seed

set_seed(42)

# Load datasets
train_df = pd.read_csv('data/train_preprocessed.csv')
aug_df   = pd.read_csv('data/train_augmented.csv')

print(f'Original train : {len(train_df)} samples')
print(f'Augmented train: {len(aug_df)} samples (+{len(aug_df)-len(train_df)})')

print(f"\n{'Aspect':<40} {'Before':>8} {'After':>8} {'Delta':>8}")
print('-' * 68)
for asp in RARE_ASPECTS:
    before = int((train_df[asp] > 0).sum())
    after  = int((aug_df[asp] > 0).sum())
    print(f'{asp:<40} {before:>8} {after:>8} {after-before:>+8}')

In [ ]:
# ============================================================
# Cell 5 — Verify EDA config (class weights, encoder config)
# ============================================================
enc_cfg = json.load(open('outputs/eda/encoder_config.json'))
print('=== Encoder Config ===')
for k, v in enc_cfg.items():
    print(f'  {k}: {v}')

cw = json.load(open('outputs/eda/class_weights.json'))
print('\n=== Global Class Weights ===')
label_map = {'0': 'absent', '1': 'positive', '2': 'negative', '3': 'neutral'}
for cls, w in cw['global_weights'].items():
    note = ' <- clip 10.0' if float(w) > 10 else ''
    print(f"  {label_map.get(cls,cls):12s}: {float(w):.1f}x{note}")

print('\nConfig OK')

In [ ]:
# ============================================================
# Cell 6 — Re-train PhoBERT tren augmented data  [CAN GPU]
# ============================================================
import torch
torch.cuda.empty_cache()
print(f'VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.1f} GB')

from transformers import AutoTokenizer
from train import train as train_fn, load_class_weights
from model import ABSAPhoBERT
from step2_dataloader import create_dataloaders

set_seed(42)

SAVE_DIR = 'outputs/results/week4_augmented'
os.makedirs(f'{SAVE_DIR}/models', exist_ok=True)

config = {
    'learning_rate':           3e-5,
    'warmup_ratio':            0.15,
    'batch_size':              16,
    'grad_accumulation_steps': 1,
    'max_epochs':              35,
    'early_stop_patience':     7,
    'dropout':                 0.2,
    'optimizer':               'AdamW',
    'scheduler':               'cosine_warmup',
    'seed':                    42,
    'max_seq_len':             256,
    'weight_clip':             10.0,
    'encoder_option':          'cls_only',
    'max_grad_norm':           1.0,
}

device = torch.device('cuda')

tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')

train_loader, dev_loader, test_loader = create_dataloaders(
    train_path='data/train_augmented.csv',
    dev_path='data/dev_preprocessed.csv',
    test_path='data/test_preprocessed.csv',
    tokenizer=tokenizer,
    batch_size=config['batch_size'],
    max_len=config['max_seq_len'],
    num_workers=2,
    use_preprocessed=True,
)

model = ABSAPhoBERT(
    model_name='vinai/phobert-base-v2',
    encoder_option=config['encoder_option'],
    dropout=config['dropout'],
).to(device)

class_weights = load_class_weights(
    weights_path='outputs/eda/class_weights.json',
    weight_clip=config['weight_clip'],
    device=device,
)

test_metrics = train_fn(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,
    class_weights=class_weights,
    device=device,
    config=config,
    save_dir=f'{SAVE_DIR}/models',
    results_dir=SAVE_DIR,
)

print('\n' + '='*55)
print('WEEK 4 AUGMENTED — RESULTS')
print(f"  ACD F1:      {test_metrics['macro_acd_f1']:.4f}")
print(f"  SPC F1:      {test_metrics['macro_spc_f1']:.4f}")
print(f"  Combined F1: {test_metrics['macro_combined_f1']:.4f}")
print('='*55)

In [ ]:
# ============================================================
# Cell 7 — So sanh ket qua baseline vs augmented
# ============================================================
import json, os

# Tim metrics files week 2 baseline (cls_only)
w2_candidates = [
    'outputs/results/week2_results_VNcoreNLP/models_cls_only/week2_test_metrics.json',
    'outputs/results/week2_results_NO_ VNcoreNLP/models_cls_only/week2_test_metrics.json',
    'outputs/results_cls_only/week2_test_metrics.json',
    'outputs/results/week2_test_metrics.json',
]
w4_candidates = [
    'outputs/results/week4_augmented/week4_test_metrics.json',
    'outputs/results/week4_augmented/test_metrics.json',
]

w2_m = next((json.load(open(p)) for p in w2_candidates if os.path.exists(p)), None)
w4_m = next((json.load(open(p)) for p in w4_candidates if os.path.exists(p)), None)

print('=== COMPARISON: Baseline vs Augmented (Test set) ===')
print(f"{'Metric':<25} {'Baseline':>12} {'Augmented':>12} {'Delta':>10}")
print('-' * 62)

for key, label in [('macro_acd_f1', 'ACD F1'), ('macro_spc_f1', 'SPC F1'),
                    ('macro_combined_f1', 'Combined F1')]:
    v2 = w2_m.get(key, 0) if w2_m else 0
    v4 = w4_m.get(key, 0) if w4_m else 0
    delta = v4 - v2
    sign = '+' if delta >= 0 else ''
    miss = ' (missing)' if not w4_m else ''
    print(f'{label:<25} {v2:>12.4f} {v4:>12.4f}{miss} {sign}{delta:>9.4f}')

if not w4_m:
    print('\nWeek4 metrics chua co — chay Cell 6 truoc.')

---
## Phase 2 — LLM Explainability

PhoBERT predict → LLM giải thích kết quả bằng tiếng Việt tự nhiên.

In [ ]:
# ============================================================
# Cell 8 — Load PhoBERT checkpoint & run inference tren test set
# ============================================================
import torch
from transformers import AutoTokenizer
from model import ABSAPhoBERT
from utils.constants import ASPECT_COLUMNS, IDX_TO_LABEL

# Tim checkpoint tot nhat co san
CHECKPOINT_CANDIDATES = [
    'outputs/results/week4_augmented/models/best_model.pt',
    'outputs/models_cls_only/best_model.pt',
    'outputs/models/best_model.pt',
]
CHECKPOINT = next((c for c in CHECKPOINT_CANDIDATES if os.path.exists(c)), None)

if not CHECKPOINT:
    raise FileNotFoundError('Khong tim thay checkpoint. Chay Cell 6 truoc.')

print(f'Loading: {CHECKPOINT}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = ABSAPhoBERT(model_name='vinai/phobert-base-v2', encoder_option='cls_only')
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model = model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')

# Run inference tren test set
test_df  = pd.read_csv('data/test_preprocessed.csv')
reviews  = test_df['Review'].tolist()
processed = test_df['processed_review'].tolist()

predictions_list = []
with torch.no_grad():
    for text in processed:
        inputs = tokenizer(
            text, max_length=256, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(input_ids=inputs['input_ids'],
                        attention_mask=inputs['attention_mask'])
        preds = outputs['preds'][0].cpu().numpy()
        pred_dict = {asp: IDX_TO_LABEL[int(preds[j])]
                     for j, asp in enumerate(ASPECT_COLUMNS)
                     if int(preds[j]) > 0}
        predictions_list.append(pred_dict)

n_with_aspects = sum(1 for p in predictions_list if p)
print(f'Inference done: {len(reviews)} reviews, {n_with_aspects} co aspect')
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Cell 9 — LLM Explain 5 reviews (demo)
# ============================================================
import time, json
from llm_client import LLMClient
from explainer import explain_predictions
from run_demo import format_output

client = LLMClient(provider=PROVIDER, api_key=API_KEY)

# Chon 5 reviews co nhieu aspects nhat
sorted_idx   = sorted(range(len(reviews)), key=lambda i: len(predictions_list[i]), reverse=True)
demo_indices = [i for i in sorted_idx if predictions_list[i]][:5]

all_results = []
for rank, idx in enumerate(demo_indices):
    review = reviews[idx]
    preds  = predictions_list[idx]

    print(f'\n[{rank+1}/5] Review {idx} ({len(preds)} aspects)...')
    explanations = explain_predictions(review, preds, client)

    result = {'review': review, 'predictions': preds, 'explanations': explanations}
    all_results.append(result)
    print(format_output(result))

    if rank < len(demo_indices) - 1:
        time.sleep(1.0)

In [ ]:
# ============================================================
# Cell 10 — Luu ket qua & Summary
# ============================================================
import shutil

# Luu explainer samples
out_path = 'outputs/results/week4_explainer_samples.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f'Saved {len(all_results)} explained reviews -> {out_path}')

# Summary
total_aspects  = sum(len(r['predictions']) for r in all_results)
total_explained = sum(len(r['explanations']) for r in all_results)
print(f'Aspects detected: {total_aspects}')
print(f'Aspects explained: {total_explained}')
if total_aspects:
    print(f'Explanation rate: {total_explained/total_aspects*100:.0f}%')

# Tao zip de download
shutil.make_archive('/kaggle/working/week4_results', 'zip', 'outputs')
print('\nZip: /kaggle/working/week4_results.zip')
print('-> Kaggle: Output panel -> Download')

print('\n=== WEEK 4 SUMMARY ===')
print('Pipeline: LLM Augmentation + PhoBERT + LLM Explainability')
if w2_m and w4_m:
    d = w4_m.get('macro_combined_f1', 0) - w2_m.get('macro_combined_f1', 0)
    print(f"Baseline Combined F1 : {w2_m.get('macro_combined_f1', 0):.4f}")
    print(f"Augmented Combined F1: {w4_m.get('macro_combined_f1', 0):.4f} ({'+' if d>=0 else ''}{d:.4f})")
elif w2_m:
    print(f"Baseline Combined F1 : {w2_m.get('macro_combined_f1', 0):.4f}")
    print('Augmented Combined F1: (chay Cell 6 truoc)')